# Computational Genetics

---

## Learning Objectives

By the end of this notebook you will be able to:

- Build the standard genetic code programmatically and translate DNA to protein
- Understand codon degeneracy and compute codon usage statistics (RSCU, CAI)
- Perform virtual restriction enzyme digests and simulate gel electrophoresis
- Find open reading frames in all 6 reading frames of a nucleotide sequence
- Construct genetic maps from two-point and three-point cross data
- Compute Ts/Tv ratios and characterize mutation spectra
- Test genotype frequencies against Hardy-Weinberg expectations


---

[← Previous: Comparative Genomics](../../Tier_2_Core_Bioinformatics/12_Comparative_Genomics/01_comparative_genomics.ipynb) | [Next: Hi-C Analysis →](../14_Hi-C_Analysis/14_hic_analysis.ipynb)

---

---

## 1. The Genetic Code

The **genetic code** is the set of rules mapping triplet codons (sequences of three nucleotides)
to amino acids (or stop signals). It is:

- **Triplet**: three consecutive nucleotides specify one amino acid
- **Non-overlapping**: each nucleotide belongs to exactly one codon
- **Degenerate**: most amino acids are encoded by more than one codon
- **Nearly universal**: the same code is used in almost all organisms

```
DNA:  5'-ATG-GCC-TAC-TGA-3'
            ↓    ↓    ↓
       Met  Ala  Tyr  STOP
```


In [ ]:
# Build the standard genetic code from first principles
# Using NCBI translation table 1 (standard code)

bases = 'TCAG'

amino_acids = (
    'FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG'
)

codon_table = {}
for i, first in enumerate(bases):
    for j, second in enumerate(bases):
        for k, third in enumerate(bases):
            codon = first + second + third
            aa_index = 16 * i + 4 * j + k
            codon_table[codon] = amino_acids[aa_index]

# Show a slice of the table
print(f'Total codons: {len(codon_table)}')
print(f'Stop codons: {[c for c,a in codon_table.items() if a == "*"]}')
print()
print('Sample entries:')
sample_codons = ['ATG', 'TAA', 'TAG', 'TGA', 'GCC', 'GCT', 'GCA', 'GCG']
for codon in sample_codons:
    aa = codon_table[codon]
    label = 'STOP' if aa == '*' else aa
    print(f'  {codon} -> {label}')


### 1.1 Translating DNA to Protein

Translation reads the coding strand 5'→3', splitting it into non-overlapping triplets,
starting at the first ATG and terminating at the first in-frame stop codon.


In [ ]:
def translate(dna_sequence, codon_table, start_codon='ATG'):
    """Translate a DNA sequence to a protein string.
    
    Starts at the first occurrence of start_codon; stops at the first
    in-frame stop codon.  Returns the protein (without the stop character).
    """
    seq = dna_sequence.upper().replace('U', 'T')
    start_pos = seq.find(start_codon)
    if start_pos == -1:
        return ''
    orf = seq[start_pos:]
    protein_residues = []
    for i in range(0, len(orf) - 2, 3):
        codon = orf[i:i+3]
        if len(codon) < 3:
            break
        aa = codon_table.get(codon, '?')
        if aa == '*':
            break
        protein_residues.append(aa)
    return ''.join(protein_residues)


# Example: a short synthetic coding sequence
coding_seq = 'GCTATGAAAGCCTTCGGTCTGATTGAATAA'
protein = translate(coding_seq, codon_table)
print(f'DNA:     {coding_seq}')
print(f'Protein: {protein}')
print(f'Length:  {len(protein)} aa')


### 1.2 Codon Degeneracy

Not all codon positions contribute equally to amino acid identity:

| Fold | Description | Example |
|------|-------------|--------|
| **1-fold** | Every change at that position changes the amino acid | AUG (Met) — only codon |
| **2-fold** | Two alternatives at the 3rd position encode the same aa | AAA/AAG → Lys |
| **4-fold** | All four bases at the 3rd position encode the same aa | GCN → Ala |

The third codon position is most often degenerate — this is why synonymous substitutions
accumulate faster than non-synonymous ones.


In [ ]:
from collections import defaultdict

# Group codons by amino acid
aa_to_codons = defaultdict(list)
for codon, aa in codon_table.items():
    aa_to_codons[aa].append(codon)

# Classify degeneracy of the 3rd position for each synonymous family
degeneracy_counts = defaultdict(int)  # fold -> number of amino acid families
for aa, codons in aa_to_codons.items():
    if aa == '*':
        continue
    fold = len(codons)
    degeneracy_counts[fold] += 1

print('Codon family sizes (fold-degeneracy) for the 20 amino acids:')
for fold in sorted(degeneracy_counts):
    print(f'  {fold}-fold: {degeneracy_counts[fold]} amino acids')

print()
print('4-fold degenerate amino acids (any nucleotide at position 3):')
for aa, codons in sorted(aa_to_codons.items()):
    if len(codons) == 4 and aa != '*':
        print(f'  {aa}: {codons}')


### 1.3 Start and Stop Codons

- **Standard start codon**: `ATG` (Met) — universal across domains of life
- **Alternative starts**: `TTG`, `GTG`, `CTG` used in bacteria (still incorporate Met)
- **Stop codons**: `TAA` (ochre), `TAG` (amber), `TGA` (opal/umber)

In some organisms stop codons are reassigned (e.g., `TGA` → Sec in selenoproteins,
`TAG` → Pyl in methanogenic archaea).


In [ ]:
# Demonstrate alternative start codons in bacteria
alt_starts = ['ATG', 'TTG', 'GTG', 'CTG']
stop_codons = [c for c, a in codon_table.items() if a == '*']

print('Start codons and their standard amino acid translation:')
for codon in alt_starts:
    aa = codon_table[codon]
    note = ' <- canonical' if codon == 'ATG' else ' <- alt start (bacteria)'
    print(f'  {codon} -> {aa}{note}')

print()
print('Stop codons (traditional names):')
stop_names = {'TAA': 'ochre', 'TAG': 'amber', 'TGA': 'opal'}
for codon in stop_codons:
    print(f'  {codon} ({stop_names[codon]})')


### Exercise 1.1: Implement `translate()` with Frame Support

Extend the `translate()` function to accept a `frame` parameter (0, 1, or 2)
that shifts the reading frame before searching for the start codon.
Then translate the sequence below in all three forward frames.

```python
test_seq = 'TAATGCCCGAATTTGCCTAAATGGGCAAATAG'
# Frame 0: starts at position 0
# Frame 1: starts at position 1
# Frame 2: starts at position 2
```


In [ ]:
# Solution
def translate_frame(dna_sequence, codon_table, frame=0):
    """Translate from a given reading frame (0, 1, or 2)."""
    return translate(dna_sequence[frame:], codon_table)


test_seq = 'TAATGCCCGAATTTGCCTAAATGGGCAAATAG'
print(f'Sequence: {test_seq} (len={len(test_seq)})')
print()
for frame in range(3):
    protein = translate_frame(test_seq, codon_table, frame)
    print(f'Frame {frame}: {protein if protein else "(no ORF found)"}')


---

## 2. Codon Usage Bias

Even though synonymous codons encode the same amino acid, organisms do not use them equally.
This **codon usage bias** arises from:

- **tRNA abundance**: abundant tRNAs match preferred codons → faster elongation
- **GC content**: genomic GC bias pushes toward GC-rich or AT-rich codons
- **Translational selection**: highly expressed genes show stronger bias
- **Mutational pressure**: background mutation rates shape the baseline

Practical consequence: heterologous protein expression benefits from **codon optimisation**
to match the host organism's preferences.


In [ ]:
from collections import Counter

def count_codons(coding_sequence):
    """Count triplet codons in a coding sequence (must start at codon position 0)."""
    seq = coding_sequence.upper()
    counts = Counter()
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        if len(codon) == 3 and 'N' not in codon:
            counts[codon] += 1
    return counts


# Synthetic E. coli-like coding sequence (200 codons)
import random
random.seed(42)

# E. coli strongly prefers certain Leu codons: CTG >> others
ecoli_codon_weights = {
    # Leu
    'CTG': 0.50, 'CTT': 0.10, 'CTC': 0.10, 'CTA': 0.04, 'TTA': 0.13, 'TTG': 0.13,
    # Ala
    'GCT': 0.18, 'GCC': 0.26, 'GCA': 0.23, 'GCG': 0.33,
    # Val
    'GTG': 0.37, 'GTT': 0.28, 'GTC': 0.20, 'GTA': 0.15,
    # Gly
    'GGC': 0.34, 'GGT': 0.35, 'GGG': 0.15, 'GGA': 0.11, 'GGN': 0.05,
}

# Build a realistic synthetic gene for E. coli
ecoli_synonymous_families = {
    'Leu': ['CTG', 'CTT', 'CTC', 'CTA', 'TTA', 'TTG'],
    'Ala': ['GCT', 'GCC', 'GCA', 'GCG'],
    'Arg': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'Gly': ['GGT', 'GGC', 'GGA', 'GGG'],
    'Pro': ['CCT', 'CCC', 'CCA', 'CCG'],
    'Thr': ['ACT', 'ACC', 'ACA', 'ACG'],
    'Val': ['GTT', 'GTC', 'GTA', 'GTG'],
    'Ser': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
}

ecoli_aa_weights = {
    'Leu': [0.50, 0.10, 0.10, 0.04, 0.13, 0.13],
    'Ala': [0.18, 0.26, 0.23, 0.33],
    'Arg': [0.36, 0.36, 0.07, 0.11, 0.07, 0.04],
    'Gly': [0.35, 0.34, 0.11, 0.15, 0.05],
    'Pro': [0.16, 0.10, 0.20, 0.55],
    'Thr': [0.19, 0.40, 0.17, 0.25],
    'Val': [0.28, 0.20, 0.15, 0.37],
    'Ser': [0.15, 0.15, 0.14, 0.15, 0.15, 0.26],
}

def build_synthetic_gene(aa_families, aa_weights, n_codons=200):
    aas = list(aa_families.keys())
    gene_codons = []
    for _ in range(n_codons):
        aa = random.choice(aas)
        codons = aa_families[aa]
        weights = aa_weights[aa][:len(codons)]
        chosen = random.choices(codons, weights=weights)[0]
        gene_codons.append(chosen)
    return 'ATG' + ''.join(gene_codons) + 'TAA'

ecoli_gene = build_synthetic_gene(ecoli_synonymous_families, ecoli_aa_weights)
ecoli_counts = count_codons(ecoli_gene)

print(f'E. coli synthetic gene length: {len(ecoli_gene)} nt ({len(ecoli_gene)//3} codons)')
print('Top 10 most frequent codons:')
for codon, count in ecoli_counts.most_common(10):
    aa = codon_table.get(codon, '?')
    print(f'  {codon} ({aa}): {count}')


### 2.1 Relative Synonymous Codon Usage (RSCU)

RSCU normalises codon counts by the expected frequency if all synonymous codons
were used equally:

$$\text{RSCU}_{ij} = \frac{X_{ij}}{\bar{X}_i} = \frac{X_{ij}}{\frac{1}{n_i}\sum_{j=1}^{n_i} X_{ij}}$$

where $X_{ij}$ is the count of the $j$-th codon for amino acid $i$, and $n_i$ is the
number of synonymous codons for that amino acid.

- **RSCU = 1.0**: codon used as expected under equal usage
- **RSCU > 1.0**: over-represented (preferred)
- **RSCU < 1.0**: under-represented (disfavoured)


In [ ]:
def compute_rscu(codon_counts, codon_table):
    """Compute RSCU for each codon given observed counts."""
    # Group codons by amino acid
    aa_groups = defaultdict(list)
    for codon, aa in codon_table.items():
        if aa != '*':
            aa_groups[aa].append(codon)

    rscu = {}
    for aa, synonyms in aa_groups.items():
        total = sum(codon_counts.get(c, 0) for c in synonyms)
        n = len(synonyms)
        expected = total / n if n > 0 else 0
        for codon in synonyms:
            observed = codon_counts.get(codon, 0)
            rscu[codon] = (observed / expected) if expected > 0 else 0.0
    return rscu


ecoli_rscu = compute_rscu(ecoli_counts, codon_table)

# Show RSCU for Leucine codons
leu_codons = ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG']
print('RSCU for Leucine (Leu) codons in E. coli synthetic gene:')
print(f'{"Codon":8} {"Count":8} {"RSCU":8}')
print('-' * 26)
for codon in leu_codons:
    count = ecoli_counts.get(codon, 0)
    rscu_val = ecoli_rscu.get(codon, 0)
    bar = '*' * int(rscu_val * 5)
    print(f'{codon:8} {count:8} {rscu_val:8.2f}  {bar}')


### 2.2 Codon Adaptation Index (CAI)

The **CAI** (Sharp & Li, 1987) measures how well a gene's codon usage matches the organism's
highly expressed reference gene set. It uses the **Relative Adaptiveness** ($w_{ij}$):

$$w_{ij} = \frac{\text{RSCU}_{ij}}{\text{RSCU}_{i,\max}}$$

The CAI for a gene of $L$ codons is the geometric mean of the $w$ values:

$$\text{CAI} = \left(\prod_{k=1}^{L} w_k\right)^{1/L}$$

CAI ranges from 0 (poorly adapted) to 1.0 (perfectly adapted to the reference).


In [ ]:
import math

def compute_cai(gene_sequence, rscu_reference, codon_table):
    """Compute CAI for a gene given a reference RSCU dictionary."""
    # Compute max RSCU per amino acid from the reference
    aa_groups = defaultdict(list)
    for codon, aa in codon_table.items():
        if aa != '*':
            aa_groups[aa].append(codon)

    max_rscu = {}
    for aa, synonyms in aa_groups.items():
        max_rscu[aa] = max(rscu_reference.get(c, 0) for c in synonyms)

    # Relative adaptiveness w for each codon
    w = {}
    for codon, aa in codon_table.items():
        if aa == '*':
            continue
        rscu_val = rscu_reference.get(codon, 0)
        w[codon] = rscu_val / max_rscu[aa] if max_rscu[aa] > 0 else 0

    # Geometric mean over the gene's codons
    gene_w_values = []
    seq = gene_sequence.upper()
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = codon_table.get(codon, '?')
        if aa in ('*', '?') or codon not in w:
            continue
        wi = w[codon]
        if wi > 0:
            gene_w_values.append(math.log(wi))

    if not gene_w_values:
        return 0.0
    return math.exp(sum(gene_w_values) / len(gene_w_values))


cai_ecoli_self = compute_cai(ecoli_gene, ecoli_rscu, codon_table)
print(f'CAI of E. coli gene vs its own RSCU: {cai_ecoli_self:.4f}')
print('(Self-CAI of a codon-optimised gene is close to 1.0)')

# Build a random gene for comparison
random_gene = 'ATG' + ''.join(random.choices('ACGT', k=600)) + 'TAA'
cai_random = compute_cai(random_gene, ecoli_rscu, codon_table)
print(f'CAI of random-sequence gene vs E. coli RSCU: {cai_random:.4f}')
print('(Random genes score poorly because they ignore codon preferences)')


### 2.3 GC Content at Codon Positions

Organisms under different mutational pressures show distinct GC content at each codon position:

- **GC1**: position 1 (first base) — partially constrained by amino acid
- **GC2**: position 2 (second base) — most constrained, changes usually alter amino acid
- **GC3**: position 3 (third base) — least constrained, best proxy for genomic GC bias

E. coli has high GC3 (~55%). *Plasmodium falciparum* (malaria parasite) has very low GC3 (~12%)
due to extreme AT bias.


In [ ]:
def gc_content_by_position(coding_sequence):
    """Compute GC% at each of the three codon positions."""
    seq = coding_sequence.upper()
    positions = [[], [], []]
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        if len(codon) == 3:
            for pos in range(3):
                positions[pos].append(codon[pos])

    gc_by_pos = {}
    for pos in range(3):
        bases = positions[pos]
        gc = sum(1 for b in bases if b in 'GC')
        gc_by_pos[pos + 1] = gc / len(bases) if bases else 0
    return gc_by_pos


ecoli_gc = gc_content_by_position(ecoli_gene)

# Simulate a human-like gene (high GC overall, GC3 ~60%)
human_families = ecoli_synonymous_families  # same families, different weights
human_aa_weights = {
    'Leu': [0.07, 0.13, 0.13, 0.20, 0.07, 0.40],  # CTG dominant
    'Ala': [0.28, 0.40, 0.15, 0.17],
    'Arg': [0.08, 0.19, 0.11, 0.21, 0.20, 0.21],
    'Gly': [0.16, 0.34, 0.25, 0.25],
    'Pro': [0.28, 0.33, 0.27, 0.11],
    'Thr': [0.24, 0.36, 0.28, 0.12],
    'Val': [0.18, 0.24, 0.12, 0.46],
    'Ser': [0.15, 0.22, 0.15, 0.06, 0.15, 0.27],
}
human_gene = build_synthetic_gene(human_families, human_aa_weights)
human_gc = gc_content_by_position(human_gene)

print(f'{"Position":12} {"E. coli GC%":14} {"Human GC%":12}')
print('-' * 40)
for pos in [1, 2, 3]:
    label = f'GC{pos}'
    print(f'{label:12} {ecoli_gc[pos]*100:14.1f} {human_gc[pos]*100:12.1f}')


---

## 3. Restriction Enzyme Analysis

Restriction enzymes (restriction endonucleases) cut double-stranded DNA at or near specific
recognition sequences. They are fundamental tools in molecular biology.

| Type | Characteristics | Example |
|------|----------------|--------|
| **Type I** | Cuts far from recognition site, requires ATP | EcoKI |
| **Type II** | Cuts within or near recognition site, simple | EcoRI, HindIII |
| **Type III** | Cuts 25-27 bp downstream | EcoPI |

Type II enzymes are used in virtually all molecular cloning because their
cut sites are predictable.


In [ ]:
# Define Type II restriction enzymes
# Format: name -> (recognition_sequence, cut_position_on_top_strand)
# cut_position is 0-indexed from the start of the recognition sequence
# A negative position means the cut is on the complement strand offset

restriction_enzymes = {
    'EcoRI':  {'site': 'GAATTC', 'cut': 1},   # G|AATTC -> 5' overhang AATT
    'BamHI':  {'site': 'GGATCC', 'cut': 1},   # G|GATCC -> 5' overhang GATC
    'HindIII':{'site': 'AAGCTT', 'cut': 1},   # A|AGCTT -> 5' overhang AGCT
    'SmaI':   {'site': 'CCCGGG', 'cut': 3},   # CCC|GGG -> blunt end
    'PstI':   {'site': 'CTGCAG', 'cut': 5},   # CTGCA|G -> 3' overhang ACGT
    'NotI':   {'site': 'GCGGCCGC', 'cut': 2}, # GC|GGCCGC -> 5' overhang GGCC
    'EcoRV':  {'site': 'GATATC', 'cut': 3},   # GAT|ATC -> blunt end
    'NcoI':   {'site': 'CCATGG', 'cut': 1},   # C|CATGG -> 5' overhang CATG
}

print('Restriction enzyme properties:')
print(f'{"Enzyme":10} {"Site":10} {"Site len":10} {"Cut pos":10}')
print('-' * 42)
for name, info in restriction_enzymes.items():
    site_len = len(info['site'])
    print(f'{name:10} {info["site"]:10} {site_len:10} {info["cut"]:10}')


### 3.1 Palindromic Recognition Sites

Most Type II recognition sequences are **palindromes** — the top strand read 5'→3' equals
the bottom strand read 5'→3'. This allows the enzyme to bind and cut symmetrically.

```
EcoRI:
5'---G A A T T C---3'
3'---C T T A A G---5'
         ^
   cut between G and A on both strands

After digestion:
5'---G     AATTC---3'
3'---CTTAA     G---5'
     \___5' overhangs___/
```


In [ ]:
def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    complement = str.maketrans('ACGTacgt', 'TGCAtgca')
    return seq.translate(complement)[::-1]


def is_palindrome(sequence):
    """Check if a DNA sequence is palindromic (equals its reverse complement)."""
    return sequence.upper() == reverse_complement(sequence).upper()


print('Checking palindrome status of recognition sites:')
print(f'{"Enzyme":10} {"Site":10} {"Palindrome?":12}')
print('-' * 34)
for name, info in restriction_enzymes.items():
    site = info['site']
    palin = is_palindrome(site)
    print(f'{name:10} {site:10} {"Yes" if palin else "No":12}')


### 3.2 Virtual Digest

A **virtual digest** computes the fragments that would be produced if a DNA molecule
were cut with one or more restriction enzymes, without actually running a lab reaction.
This is essential for:

- Selecting enzymes for cloning (choosing enzymes that produce compatible ends)
- Designing restriction maps
- Verifying plasmid constructs in silico before ordering DNA


In [ ]:
def find_cut_sites(sequence, enzyme_info):
    """Find all cut positions for one enzyme in a linear sequence.
    
    Returns a list of (cut_position, enzyme_name) tuples.
    cut_position is the absolute index in the sequence where the cut occurs.
    """
    site = enzyme_info['site']
    cut_offset = enzyme_info['cut']
    seq_upper = sequence.upper()
    cuts = []
    start = 0
    while True:
        pos = seq_upper.find(site, start)
        if pos == -1:
            break
        cut_pos = pos + cut_offset
        cuts.append(cut_pos)
        start = pos + 1
    return cuts


def virtual_digest(sequence, selected_enzymes, restriction_enzymes):
    """Perform a multi-enzyme virtual digest on a linear DNA sequence.
    
    Returns fragment sizes sorted largest to smallest (as on a gel).
    """
    all_cuts = []
    for enzyme_name in selected_enzymes:
        info = restriction_enzymes[enzyme_name]
        cuts = find_cut_sites(sequence, info)
        for cut in cuts:
            all_cuts.append((cut, enzyme_name))

    all_cuts.sort(key=lambda x: x[0])
    cut_positions = [0] + [c[0] for c in all_cuts] + [len(sequence)]

    fragments = []
    for i in range(len(cut_positions) - 1):
        size = cut_positions[i+1] - cut_positions[i]
        if size > 0:
            fragments.append(size)

    return sorted(fragments, reverse=True), all_cuts


# Synthetic 4 kb plasmid insert with known sites
import random
random.seed(7)

def build_sequence_with_sites(total_length, sites_to_embed):
    """Build a random sequence with specific restriction sites embedded."""
    filler_len = total_length - sum(len(s) for s in sites_to_embed)
    filler = ''.join(random.choices('ACGT', k=filler_len))
    # Insert sites at evenly spaced positions
    parts = []
    chunk = filler_len // (len(sites_to_embed) + 1)
    idx = 0
    for i, site in enumerate(sites_to_embed):
        parts.append(filler[idx: idx + chunk])
        parts.append(site)
        idx += chunk
    parts.append(filler[idx:])
    return ''.join(parts)


plasmid_insert = build_sequence_with_sites(
    4000,
    ['GAATTC', 'GGATCC', 'AAGCTT', 'GAATTC']  # EcoRI, BamHI, HindIII, EcoRI
)

enzymes_used = ['EcoRI', 'BamHI', 'HindIII']
fragments, cuts = virtual_digest(plasmid_insert, enzymes_used, restriction_enzymes)

print(f'Sequence length: {len(plasmid_insert)} bp')
print(f'Enzymes used: {", ".join(enzymes_used)}')
print()
print('Cut sites (position | enzyme):')
for pos, enzyme in cuts:
    print(f'  {pos:6} bp | {enzyme}')
print()
print('Predicted gel bands (largest to smallest):')
for i, frag in enumerate(fragments):
    print(f'  Band {i+1}: {frag} bp')
print(f'Total: {sum(fragments)} bp (should equal sequence length)')


### 3.3 Gel Electrophoresis Simulation

In agarose gel electrophoresis, DNA fragments migrate inversely proportional to the
log of their size. Smaller fragments travel farther from the wells.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def simulate_gel(fragment_size_lists, lane_labels, ladder_sizes=None, title='Gel Electrophoresis'):
    """Visualise a simulated agarose gel."""
    if ladder_sizes is None:
        ladder_sizes = [10000, 8000, 6000, 5000, 4000, 3000, 2000, 1500, 1000, 750, 500, 250]

    # Migration distance is proportional to log(size) relative to ladder
    max_log = math.log10(max(ladder_sizes))
    min_log = math.log10(min(ladder_sizes))

    def migration(size):
        """Normalised migration: 0 at top, 1 at bottom."""
        if size <= 0:
            return 1.0
        log_size = math.log10(max(size, 10))
        return (max_log - log_size) / (max_log - min_log)

    n_lanes = 1 + len(fragment_size_lists)  # ladder + sample lanes
    fig, ax = plt.subplots(figsize=(3 * n_lanes, 6))
    ax.set_facecolor('black')
    fig.patch.set_facecolor('#1a1a2e')

    lane_width = 0.6

    # Draw ladder
    for size in ladder_sizes:
        y = migration(size)
        ax.barh(y, lane_width, left=0.2, height=0.015, color='#00ff88', alpha=0.85)
        ax.text(0.2 - 0.05, y, f'{size}', va='center', ha='right', color='#00ff88', fontsize=7)

    # Draw sample lanes
    for lane_idx, (frags, label) in enumerate(zip(fragment_size_lists, lane_labels)):
        x_pos = 1.2 + lane_idx * 1.1
        for size in frags:
            y = migration(size)
            ax.barh(y, lane_width, left=x_pos, height=0.015, color='#4fc3f7', alpha=0.9)
            ax.text(x_pos + lane_width + 0.05, y, f'{size}', va='center', fontsize=7, color='#4fc3f7')
        ax.text(x_pos + lane_width / 2, -0.05, label, ha='center', color='white', fontsize=9)

    ax.text(0.2 + lane_width / 2, -0.05, 'Ladder', ha='center', color='#00ff88', fontsize=9)
    ax.set_xlim(-0.5, n_lanes * 1.2)
    ax.set_ylim(-0.12, 1.1)
    ax.set_title(title, color='white', pad=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


simulate_gel(
    [fragments],
    ['EcoRI+BamHI+HindIII'],
    title='Virtual Digest — Simulated Agarose Gel'
)


---

## 4. Open Reading Frame (ORF) Finding

An **Open Reading Frame** is a stretch of DNA that begins with a start codon (ATG) and
ends with an in-frame stop codon (TAA, TAG, or TGA), with no intervening stop codons.

### Why search all 6 frames?

Double-stranded DNA can be read in **6 reading frames**:

```
5'----ATGCCCGAATTTGCCTAAATGGGCAAATAG----3'
      Frame +1:  ATG-CCC-GAA-TTT-GCC-TAA...  <- ORF found
      Frame +2:   TGC-CCG-AAT-TTG-CCT-AAA...
      Frame +3:    GCC-CGA-ATT-TGC-CTA-AAT...

3'----TACGGGCTTAAACGGATTTTACCCGTTTTTAC----5'
      (= RC of top strand, also read 5'→3' in 3 frames)
```


In [ ]:
def find_orfs(sequence, codon_table, min_length=30):
    """Find all ORFs in a DNA sequence across all 6 reading frames.
    
    Parameters
    ----------
    sequence   : nucleotide string (any case)
    codon_table: dict mapping codon -> amino acid
    min_length : minimum ORF length in nucleotides (including start, excluding stop)
    
    Returns
    -------
    List of dicts with keys: start, end, frame, strand, protein
    """
    seq = sequence.upper()
    rc_seq = reverse_complement(seq)
    stop_codons = {c for c, a in codon_table.items() if a == '*'}

    def scan_strand(strand_seq, strand_label):
        orfs = []
        for frame in range(3):
            i = frame
            orf_start = None
            while i <= len(strand_seq) - 3:
                codon = strand_seq[i:i+3]
                if codon == 'ATG' and orf_start is None:
                    orf_start = i
                elif codon in stop_codons and orf_start is not None:
                    orf_len = i - orf_start
                    if orf_len >= min_length:
                        protein = ''
                        for j in range(orf_start, i, 3):
                            aa = codon_table.get(strand_seq[j:j+3], '?')
                            if aa == '*':
                                break
                            protein += aa
                        # Map coordinates back to the original sequence
                        if strand_label == '+':
                            abs_start, abs_end = orf_start, i + 3
                        else:
                            abs_end = len(seq) - orf_start
                            abs_start = len(seq) - (i + 3)
                        orfs.append({
                            'start': abs_start,
                            'end': abs_end,
                            'frame': frame + 1,
                            'strand': strand_label,
                            'length_nt': orf_len,
                            'protein': protein,
                        })
                    orf_start = None
                i += 3
        return orfs

    orfs = scan_strand(seq, '+') + scan_strand(rc_seq, '-')
    return sorted(orfs, key=lambda x: -x['length_nt'])


# Test on a short synthetic sequence with a known embedded ORF
test_dna = (
    'GCTAGCTAGCATGCCCGAATTTGCCTTAGCTAGCATG'
    'AAAGCCTTCGGTCTGATTGAATAATAGCTAGCGATC'
    'ATGCGTAAAGCCTTCGGTCTGATTCTGTAAATCGATCGATCG'
)

orfs = find_orfs(test_dna, codon_table, min_length=15)
print(f'Sequence length: {len(test_dna)} nt')
print(f'ORFs found (min 15 nt): {len(orfs)}')
print()
print(f'{"#":3} {"Strand":8} {"Frame":7} {"Start":7} {"End":7} {"Length (nt)":13} {"Protein"}')
print('-' * 70)
for idx, orf in enumerate(orfs, 1):
    print(f'{idx:3} {orf["strand"]:8} {orf["frame"]:7} {orf["start"]:7} '
          f'{orf["end"]:7} {orf["length_nt"]:13} {orf["protein"]}')


### 4.1 Visualising ORF Maps

An **ORF map** (also called a six-frame translation diagram) displays all ORFs found
across the 6 reading frames as horizontal bars. Longer bars in the same frame without
overlapping stops are more likely to represent real genes.


In [ ]:
def plot_orf_map(sequence, orfs, title='ORF Map'):
    """Draw a six-frame ORF map."""
    fig, ax = plt.subplots(figsize=(12, 4))

    strand_y = {('+', 1): 5, ('+', 2): 4, ('+', 3): 3,
                ('-', 1): 2, ('-', 2): 1, ('-', 3): 0}
    colors = {'+':	'#2196F3', '-': '#F44336'}

    for orf in orfs:
        y = strand_y.get((orf['strand'], orf['frame']), 0)
        start = min(orf['start'], orf['end'])
        end = max(orf['start'], orf['end'])
        ax.barh(y, end - start, left=start, height=0.7,
                color=colors[orf['strand']], alpha=0.8, edgecolor='white', linewidth=0.5)
        if end - start > len(sequence) * 0.04:
            ax.text((start + end) / 2, y, f'{end - start} nt',
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')

    y_labels = ['+1', '+2', '+3', '-1', '-2', '-3']
    ax.set_yticks([5, 4, 3, 2, 1, 0])
    ax.set_yticklabels(y_labels[::-1][:6])
    ax.set_xlabel('Sequence position (nt)')
    ax.set_title(title)
    ax.set_xlim(0, len(sequence))
    ax.axhline(y=2.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.text(len(sequence) * 0.01, 4.0, 'Forward (+)', color='#2196F3', fontsize=9)
    ax.text(len(sequence) * 0.01, 1.5, 'Reverse (-)', color='#F44336', fontsize=9)
    plt.tight_layout()
    plt.show()


# Build a longer sequence so the ORF map is more illustrative
random.seed(99)
genome_fragment = (
    ''.join(random.choices('ACGT', k=100))
    + 'ATGAAAGCCTTCGGTCTGATTGAACTGGTCAAAGCCTAA'  # 40 nt ORF frame+1
    + ''.join(random.choices('ACGT', k=80))
    + 'ATGCGTAAAGCCTTCGGTCTGATTCTGTAA'           # 30 nt ORF frame+1
    + ''.join(random.choices('ACGT', k=50))
)

genome_orfs = find_orfs(genome_fragment, codon_table, min_length=21)
print(f'Fragment length: {len(genome_fragment)} nt')
print(f'ORFs >= 21 nt: {len(genome_orfs)}')
plot_orf_map(genome_fragment, genome_orfs, title='Six-Frame ORF Map')


### 4.2 Identifying the Most Likely Coding ORF

Heuristics for distinguishing real genes from spurious ORFs:

1. **Length**: real bacterial ORFs are typically > 300 nt (> 100 aa)
2. **Codon usage**: real ORFs have codon usage similar to the host organism
3. **Kozak context** (eukaryotes): `(gcc)gccRccAUGG` around the start codon
4. **Shine-Dalgarno** (prokaryotes): AAGG 5-10 nt upstream of start
5. **ORF density**: in bacteria ~88% of the genome is coding; overlapping ORFs are rare


In [ ]:
# Find the longest ORF as the most likely coding sequence
if genome_orfs:
    longest_orf = genome_orfs[0]  # already sorted by length descending
    print('Most likely coding ORF (longest):')
    print(f'  Strand : {longest_orf["strand"]}')
    print(f'  Frame  : {longest_orf["frame"]}')
    print(f'  Start  : {longest_orf["start"]} nt')
    print(f'  End    : {longest_orf["end"]} nt')
    print(f'  Length : {longest_orf["length_nt"]} nt ({longest_orf["length_nt"]//3} aa)')
    print(f'  Protein: {longest_orf["protein"]}')

print()
print('Length distribution of all ORFs found:')
lengths = sorted([o['length_nt'] for o in genome_orfs])
for length in lengths:
    bar = '#' * (length // 10)
    print(f'  {length:5} nt  {bar}')


---

## 5. Genetic Mapping

**Genetic (linkage) mapping** determines the order of genes on a chromosome and the
distances between them, measured in **centimorgans (cM)** or **map units (m.u.)**.

One centimorgan ≈ 1% recombination frequency between two loci.

```
           A         B         C
   ────────●────────●────────────●────────
             10 cM    25 cM
```

**Recombination frequency (RF)** is estimated from cross data:

$$RF = \frac{\text{recombinant progeny}}{\text{total progeny}} \times 100\%$$


In [ ]:
# Two-point cross: estimating recombination frequency
# Cross: AaBb (double heterozygote) x aabb (double recessive)

two_point_data = {
    'AB': 430,   # parental class
    'ab': 440,   # parental class
    'Ab': 62,    # recombinant
    'aB': 68,    # recombinant
}

parental = two_point_data['AB'] + two_point_data['ab']
recombinant = two_point_data['Ab'] + two_point_data['aB']
total = sum(two_point_data.values())

rf_ab = recombinant / total * 100

print('Two-point cross: AaBb × aabb')
print()
print('Progeny classes:')
for genotype, count in two_point_data.items():
    pct = count / total * 100
    class_type = 'parental' if genotype in ('AB', 'ab') else 'recombinant'
    print(f'  {genotype}: {count:4d}  ({pct:.1f}%)  [{class_type}]')

print()
print(f'Total progeny: {total}')
print(f'Parental class: {parental} ({parental/total*100:.1f}%)')
print(f'Recombinant class: {recombinant} ({recombinant/total*100:.1f}%)')
print(f'Recombination frequency (RF): {rf_ab:.1f} cM')


### 5.1 Three-Point Crosses: Determining Gene Order

Three-point crosses allow us to:
1. Determine the **order** of three genes (which is in the middle)
2. Calculate **map distances** between adjacent pairs
3. Measure **interference** — whether one crossover suppresses another

The key insight: **double recombinants are the rarest class**.
The gene in the middle is identified because double recombinants swap *only* the middle gene.


In [ ]:
# Three-point cross data
# Genes a, b, c — unknown order; cross: abc/+++ x abc/abc
three_point_data = {
    '+++':    580,  # parental
    'abc':    592,  # parental
    'ab+':     85,  # single CO between c and its neighbour
    '++c':     83,  # single CO
    'a++':    218,  # single CO between a and b
    '+bc':    214,  # single CO
    'a+c':     22,  # double CO
    '+b+':     18,  # double CO  <- smallest classes = double recombinants
}

total = sum(three_point_data.values())

print('Three-point cross progeny:')
for genotype, count in three_point_data.items():
    pct = count / total * 100
    print(f'  {genotype}: {count:4d}  ({pct:.2f}%)')

print(f'\nTotal: {total}')
print()

# Double recombinants are 'a+c' and '+b+' -- b is flanked by a and c
# Therefore gene order is: a - b - c  (b is in the middle)
double_CO = three_point_data['a+c'] + three_point_data['+b+']
print(f'Double recombinants: {double_CO} ({double_CO/total*100:.2f}%)')
print('=> b is in the middle (double COs swap only the middle gene)')
print('=> Gene order: a --- b --- c')


In [ ]:
# Calculate map distances
# Region I: a to b
# Region II: b to c

# Single COs in region I (between a and b): 'a++' and '+bc'
# Single COs in region II (between b and c): 'ab+' and '++c'
# Double COs contribute to BOTH regions

single_co_region_I = three_point_data['a++'] + three_point_data['+bc']
single_co_region_II = three_point_data['ab+'] + three_point_data['++c']

# Map distance = (single COs + double COs) / total * 100
dist_a_b = (single_co_region_I + double_CO) / total * 100
dist_b_c = (single_co_region_II + double_CO) / total * 100

print('Map distances:')
print(f'  a --- b: {dist_a_b:.1f} cM')
print(f'  b --- c: {dist_b_c:.1f} cM')
print(f'  a --- c (sum): {dist_a_b + dist_b_c:.1f} cM')
print()

# Coefficient of coincidence and interference
# CoC = observed double COs / expected double COs
expected_double_CO = (dist_a_b / 100) * (dist_b_c / 100) * total
coc = double_CO / expected_double_CO
interference = 1 - coc

print(f'Expected double COs (if independent): {expected_double_CO:.1f}')
print(f'Observed double COs: {double_CO}')
print(f'Coefficient of Coincidence (CoC): {coc:.3f}')
print(f'Interference: {interference:.3f}')
print()
if interference > 0:
    print('Positive interference: one crossover SUPPRESSES nearby crossovers (common in eukaryotes)')
elif interference < 0:
    print('Negative interference: one crossover PROMOTES nearby crossovers')
else:
    print('No interference: crossovers are independent')


### 5.2 Drawing the Genetic Map


In [ ]:
def draw_genetic_map(gene_order, distances, title='Genetic Map'):
    """Draw a simple linear genetic map."""
    fig, ax = plt.subplots(figsize=(10, 2.5))

    # Compute cumulative positions
    positions = [0]
    for d in distances:
        positions.append(positions[-1] + d)
    total_map_len = positions[-1]

    # Draw chromosome line
    ax.plot([0, total_map_len], [0.5, 0.5], 'k-', linewidth=4, solid_capstyle='round')

    # Draw gene markers
    for gene, pos in zip(gene_order, positions):
        ax.plot(pos, 0.5, 'o', color='#e91e63', markersize=14, zorder=5)
        ax.text(pos, 0.5, gene, ha='center', va='center', fontsize=11,
                color='white', fontweight='bold', zorder=6)
        ax.text(pos, 0.2, f'{pos:.0f} cM', ha='center', va='top', fontsize=9, color='#333')

    # Draw distance brackets
    for i, d in enumerate(distances):
        mid = (positions[i] + positions[i+1]) / 2
        ax.annotate('', xy=(positions[i+1], 0.78), xytext=(positions[i], 0.78),
                    arrowprops=dict(arrowstyle='<->', color='#555', lw=1.5))
        ax.text(mid, 0.85, f'{d:.1f} cM', ha='center', va='bottom', fontsize=9, color='#555')

    ax.set_xlim(-5, total_map_len + 5)
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


draw_genetic_map(
    gene_order=['a', 'b', 'c'],
    distances=[dist_a_b, dist_b_c],
    title='Three-Point Cross: Genetic Map (a — b — c)'
)


---

## 6. Mutation Analysis

### Transitions and Transversions

Point mutations are classified by the chemical nature of the change:

| Type | Definition | Examples |
|------|-----------|--------|
| **Transition (Ts)** | Purine ↔ Purine or Pyrimidine ↔ Pyrimidine | A↔G, C↔T |
| **Transversion (Tv)** | Purine ↔ Pyrimidine | A↔C, A↔T, G↔C, G↔T |

Although there are twice as many possible transversions, **transitions are more frequent**
in most genomes (Ts/Tv ratio typically 2–4 in mammalian genomes).
CpG sites have a very high transition rate due to spontaneous deamination of 5-methylcytosine.


In [ ]:
def classify_substitution(ref_base, alt_base):
    """Classify a single nucleotide substitution as Ts or Tv."""
    purines = {'A', 'G'}
    pyrimidines = {'C', 'T'}
    ref = ref_base.upper()
    alt = alt_base.upper()
    if ref == alt:
        return 'no change'
    if (ref in purines and alt in purines) or (ref in pyrimidines and alt in pyrimidines):
        return 'Ts'
    return 'Tv'


# Enumerate all possible substitutions
all_bases = 'ACGT'
print('All possible substitution types:')
print(f'{"Ref→Alt":10} {"Type"}')
print('-' * 22)
for ref in all_bases:
    for alt in all_bases:
        if ref != alt:
            stype = classify_substitution(ref, alt)
            print(f'{ref}->{alt:6} {stype}')

ts_count = sum(1 for r in all_bases for a in all_bases
               if r != a and classify_substitution(r, a) == 'Ts')
tv_count = sum(1 for r in all_bases for a in all_bases
               if r != a and classify_substitution(r, a) == 'Tv')
print(f'\nTotal Ts possible: {ts_count}')
print(f'Total Tv possible: {tv_count}')
print(f'Tv/Ts ratio by chance: {tv_count/ts_count:.1f}')


### 6.1 Computing Ts/Tv from Aligned Sequences


In [ ]:
def compute_tstv(seq1, seq2):
    """Compute Ts, Tv, and Ts/Tv ratio from two aligned sequences."""
    assert len(seq1) == len(seq2), 'Sequences must be same length (aligned)'
    ts = tv = 0
    for r, a in zip(seq1.upper(), seq2.upper()):
        if r == '-' or a == '-' or r == 'N' or a == 'N':
            continue
        stype = classify_substitution(r, a)
        if stype == 'Ts':
            ts += 1
        elif stype == 'Tv':
            tv += 1
    tstv = ts / tv if tv > 0 else float('inf')
    return {'Ts': ts, 'Tv': tv, 'Ts/Tv': tstv, 'total_SNPs': ts + tv}


# Generate two synthetic sequences with a realistic Ts/Tv ratio
random.seed(2024)
ref_seq = ''.join(random.choices('ACGT', k=1000))

# Introduce mutations with ~3:1 Ts:Tv bias
ts_substitutions = {'A': 'G', 'G': 'A', 'C': 'T', 'T': 'C'}
tv_substitutions = {'A': 'C', 'G': 'T', 'C': 'A', 'T': 'G'}

mutated_seq = list(ref_seq)
mutation_rate = 0.05
for i in range(len(mutated_seq)):
    if random.random() < mutation_rate:
        base = mutated_seq[i]
        if random.random() < 0.75:  # 75% transitions
            mutated_seq[i] = ts_substitutions[base]
        else:
            mutated_seq[i] = tv_substitutions[base]
mutated_seq = ''.join(mutated_seq)

result = compute_tstv(ref_seq, mutated_seq)
print(f'Aligned length: {len(ref_seq)} bp')
print(f'Transitions (Ts): {result["Ts"]}')
print(f'Transversions (Tv): {result["Tv"]}')
print(f'Total SNPs: {result["total_SNPs"]}')
print(f'SNP rate: {result["total_SNPs"] / len(ref_seq) * 100:.2f}%')
print(f'Ts/Tv ratio: {result["Ts/Tv"]:.3f}')


### 6.2 Mutation Spectrum

The **mutation spectrum** shows the frequency of each substitution type (e.g., C>T, A>G…).
It is a fingerprint of the mutational processes acting on a genome:

- **C>T at CpG** — deamination of 5-methylcytosine (most common human germline mutation)
- **C>A** — oxidative damage (8-oxoguanine)
- **C>G** — strand-coordinated mutations from APOBEC activity
- **T>A** — strand slippage, some chemical mutagens


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def mutation_spectrum(seq1, seq2):
    """Count each type of substitution between two aligned sequences.
    
    Changes are reported on the pyrimidine base (C or T context) by convention.
    """
    pyrimidine_map = {'A': 'T', 'G': 'C'}  # complement for purines
    counts = Counter()
    for r, a in zip(seq1.upper(), seq2.upper()):
        if r == a or r in '-N' or a in '-N':
            continue
        # Normalise to pyrimidine reference
        if r in 'AG':
            r, a = pyrimidine_map[r], pyrimidine_map.get(a, a)
        change = f'{r}>{a}'
        counts[change] += 1
    return counts


spectrum = mutation_spectrum(ref_seq, mutated_seq)

# Plot the spectrum
change_types = ['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']
colors_spectrum = ['#03A9F4', '#333', '#F44336', '#999', '#4CAF50', '#FF9800']
counts_list = [spectrum.get(c, 0) for c in change_types]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(change_types, counts_list, color=colors_spectrum, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Substitution type (pyrimidine convention)')
ax.set_ylabel('Count')
ax.set_title('Mutation Spectrum')
for bar, count in zip(bars, counts_list):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print('Mutation spectrum (pyrimidine context):')
for change, count in sorted(spectrum.items()):
    print(f'  {change}: {count}')


### 6.3 CpG Dinucleotide Deamination

In mammals, cytosine at **CpG dinucleotides** is often methylated (5mC). Spontaneous
deamination converts 5mC → T, producing a G:T mismatch. If unrepaired:

```
5'---CpG---3'   methylation    5'---C(me)pG---3'   deamination    5'---TpG---3'
3'---GpC---5'    →             3'---GpC(me)---5'   →              3'---GpC---5'
                                                      after replication: CpG → TpG / CpA
```

This explains why CpG dinucleotides are **5x under-represented** in mammalian genomes:
they mutate away faster than they arise.


In [ ]:
def cpg_observed_expected(sequence):
    """Compute CpG observed/expected ratio.
    
    O/E > 0.6 indicates a CpG island (methylation-free, gene regulatory region).
    O/E < 0.2 indicates global methylation / CpG depletion.
    """
    seq = sequence.upper()
    n = len(seq)
    c_count = seq.count('C')
    g_count = seq.count('G')
    cpg_count = sum(1 for i in range(len(seq) - 1) if seq[i:i+2] == 'CG')

    if c_count == 0 or g_count == 0:
        return 0.0
    expected = (c_count / n) * (g_count / n) * (n - 1)
    return cpg_count / expected if expected > 0 else 0


# Compare CpG O/E in three contexts:
# 1. Random sequence (baseline)
random.seed(5)
rand_seq = ''.join(random.choices('ACGT', k=1000))

# 2. Simulate globally methylated region (CpG depleted: CG->TG transitions applied)
depleted = rand_seq
for _ in range(80):  # apply many CG->TG deamination events
    idx = random.randint(0, len(depleted) - 2)
    if depleted[idx:idx+2] == 'CG':
        depleted = depleted[:idx] + 'TG' + depleted[idx+2:]

# 3. CpG island (insert extra CG dinucleotides in a 200 bp window)
cpg_island_core = ''.join(random.choices(['CG', 'GC', 'CC', 'GG'], k=100))
cpg_island_seq = rand_seq[:200] + cpg_island_core + rand_seq[200:800]

print('CpG Observed/Expected ratios:')
print(f'  Random sequence:           {cpg_observed_expected(rand_seq):.3f}')
print(f'  CpG-depleted (methylated): {cpg_observed_expected(depleted):.3f}')
print(f'  CpG island region:         {cpg_observed_expected(cpg_island_seq):.3f}')
print()
print('Threshold: O/E > 0.6 suggests a CpG island (unmethylated promoter region)')


---

## 7. Hardy-Weinberg Equilibrium (HWE)

In a large, randomly mating population with no selection, mutation, migration, or
genetic drift, allele and genotype frequencies remain constant from generation to generation.
This is **Hardy-Weinberg Equilibrium**.

For a biallelic locus with alleles A (frequency $p$) and a (frequency $q$), where $p + q = 1$:

$$P(AA) = p^2 \qquad P(Aa) = 2pq \qquad P(aa) = q^2$$

| Genotype | Expected frequency | Interpretation |
|----------|-------------------|---------------|
| AA | $p^2$ | homozygous dominant |
| Aa | $2pq$ | heterozygous |
| aa | $q^2$ | homozygous recessive |


In [ ]:
def estimate_allele_frequencies(n_AA, n_Aa, n_aa):
    """Estimate p and q from observed genotype counts."""
    n_total = n_AA + n_Aa + n_aa
    n_alleles = 2 * n_total
    p = (2 * n_AA + n_Aa) / n_alleles
    q = 1 - p
    return p, q


def hwe_expected(p, q, n_total):
    """Compute expected genotype counts under HWE."""
    return {
        'AA': p ** 2 * n_total,
        'Aa': 2 * p * q * n_total,
        'aa': q ** 2 * n_total,
    }


# Observed genotype counts at an SNP locus
observed = {'AA': 320, 'Aa': 480, 'aa': 200}
n_total = sum(observed.values())

p, q = estimate_allele_frequencies(observed['AA'], observed['Aa'], observed['aa'])
expected = hwe_expected(p, q, n_total)

print(f'Sample size: {n_total} individuals')
print(f'Allele frequencies: p(A) = {p:.4f}, q(a) = {q:.4f}')
print()
print(f'{"Genotype":10} {"Observed":12} {"Expected":12} {"Obs freq":12} {"Exp freq"}')
print('-' * 60)
for gt in ['AA', 'Aa', 'aa']:
    obs = observed[gt]
    exp = expected[gt]
    print(f'{gt:10} {obs:12} {exp:12.1f} {obs/n_total:12.4f} {exp/n_total:.4f}')


### 7.1 Chi-Squared Goodness-of-Fit Test for HWE

We test whether observed genotype counts deviate significantly from HWE expectations
using the chi-squared statistic with **1 degree of freedom** (3 genotype classes − 2
estimated parameters − 1 = 1):

$$\chi^2 = \sum_i \frac{(O_i - E_i)^2}{E_i}$$

A significant result ($p < 0.05$) indicates a departure from HWE, suggesting selection,
population stratification, inbreeding, or genotyping error.


In [ ]:
import math

def chi_squared_hwe(observed_genotypes, expected_genotypes, df=1):
    """Chi-squared goodness-of-fit test for Hardy-Weinberg equilibrium.
    
    Returns chi2 statistic and p-value (computed from chi2 CDF approximation).
    """
    chi2 = 0.0
    for gt in observed_genotypes:
        obs = observed_genotypes[gt]
        exp = expected_genotypes[gt]
        if exp > 0:
            chi2 += (obs - exp) ** 2 / exp

    # P-value from chi2 distribution using the regularised incomplete gamma function
    # math.gammainc is available in Python 3.11+; use scipy if available
    try:
        from scipy.stats import chi2 as chi2_dist
        p_value = chi2_dist.sf(chi2, df)
    except ImportError:
        # Manual approximation via incomplete gamma (Wilson-Hilferty)
        # Good enough for demonstration
        def chi2_pvalue_approx(x, k=1):
            """Very rough approximation; use scipy for production use."""
            if x <= 0:
                return 1.0
            z = (x / k) ** (1/3)
            mu = 1 - 2 / (9 * k)
            sigma = math.sqrt(2 / (9 * k))
            z_norm = (z - mu) / sigma
            # standard normal survival function approximation
            p = 0.5 * math.erfc(z_norm / math.sqrt(2))
            return p
        p_value = chi2_pvalue_approx(chi2, df)

    return chi2, p_value


chi2_stat, p_val = chi_squared_hwe(observed, expected)

print(f'Chi-squared statistic: {chi2_stat:.4f}')
print(f'Degrees of freedom: 1')
print(f'P-value: {p_val:.4f}')
print()
if p_val < 0.05:
    print('Result: SIGNIFICANT deviation from HWE (p < 0.05)')
    print('Possible causes: selection, inbreeding, population stratification, genotyping error')
else:
    print('Result: No significant deviation from HWE (p >= 0.05)')
    print('Population appears to be in Hardy-Weinberg equilibrium at this locus')


### 7.2 When HWE Breaks Down

HWE violations are informative — they signal evolutionary or demographic forces:

| Cause | Effect on genotype frequencies |
|-------|-------------------------------|
| **Positive selection** | Favoured allele increases in frequency over time |
| **Inbreeding** | Excess homozygotes relative to expected |
| **Population stratification** | Wahlund effect: apparent excess of homozygotes |
| **Genetic drift (small pop)** | Random fluctuations; fixation possible |
| **Migration** | Allele frequency change due to gene flow |
| **Non-random mating** | Assortative mating shifts heterozygosity |

For a practical implementation of HWE testing on real VCF data,
see the **`hwe_test()`** function in the
[ngs-variant-calling skill](../../../../.claude/skills/ngs-variant-calling.md).


---

## 8. Exercises


### Exercise 1: Codon Usage for a Real Gene Sequence

The sequence below is the first 300 nt of the *E. coli* **lacZ** (beta-galactosidase) CDS.
Compute:
1. The codon counts
2. RSCU for each amino acid family
3. The most over-represented codon (highest RSCU)
4. GC1, GC2, GC3 content

```python
lacZ_fragment = (
    'ATGACCATGATTACGCCAAGCTATTTTAGTGAAACTGCAATTTATTCAGCAGGAGCATTT'
    'TACCGGCATGTCATCGATACGCTGCCCAGCATTTCGAAACGCAATTTGCGCAGTTTAATG'
    'GCTTCGGCTATGATAATCGGGCAAACTATCAGTGTTTGACAGGATATATTGGCTTTGATG'
    'CCTTCGGCTATGATAATCGGGCAAACTATCAGTGTTTGACAGGATATATTGGCTTTGATG'
    'CCTTCGCTCATGAGACAATACCGG'
)
```


In [ ]:
lacZ_fragment = (
    'ATGACCATGATTACGCCAAGCTATTTTAGTGAAACTGCAATTTATTCAGCAGGAGCATTT'
    'TACCGGCATGTCATCGATACGCTGCCCAGCATTTCGAAACGCAATTTGCGCAGTTTAATG'
    'GCTTCGGCTATGATAATCGGGCAAACTATCAGTGTTTGACAGGATATATTGGCTTTGATG'
    'CCTTCGGCTATGATAATCGGGCAAACTATCAGTGTTTGACAGGATATATTGGCTTTGATG'
    'CCTTCGCTCATGAGACAATACCGG'
)

# --- Your solution here ---
lacZ_counts = count_codons(lacZ_fragment)
lacZ_rscu = compute_rscu(lacZ_counts, codon_table)
lacZ_gc = gc_content_by_position(lacZ_fragment)

# Most over-represented codon
best_codon = max(lacZ_rscu, key=lambda c: lacZ_rscu[c])

print('lacZ fragment analysis:')
print(f'  Length: {len(lacZ_fragment)} nt')
print(f'  Codons counted: {sum(lacZ_counts.values())}')
print(f'  Most over-represented codon: {best_codon} '
      f'({codon_table[best_codon]}) RSCU={lacZ_rscu[best_codon]:.2f}')
print()
print('GC content by position:')
for pos in [1, 2, 3]:
    print(f'  GC{pos}: {lacZ_gc[pos]*100:.1f}%')


### Exercise 2: Multi-Enzyme Virtual Digest

Given the sequence below, perform a virtual digest with **EcoRI + NotI**.
Predict the fragment sizes and draw the gel.

```python
cloning_vector = build_sequence_with_sites(
    6000,
    ['GAATTC', 'GCGGCCGC', 'GAATTC']  # EcoRI, NotI, EcoRI
)
```


In [ ]:
random.seed(11)
cloning_vector = build_sequence_with_sites(
    6000,
    ['GAATTC', 'GCGGCCGC', 'GAATTC']  # EcoRI, NotI, EcoRI
)

# --- Solution ---
vector_enzymes = ['EcoRI', 'NotI']
vector_fragments, vector_cuts = virtual_digest(cloning_vector, vector_enzymes, restriction_enzymes)

print(f'Vector length: {len(cloning_vector)} bp')
print(f'Digest with: {", ".join(vector_enzymes)}')
print()
print('Cut sites:')
for pos, enzyme in vector_cuts:
    print(f'  {pos:6} bp | {enzyme}')
print()
print('Predicted bands:')
for i, frag in enumerate(vector_fragments):
    print(f'  Band {i+1}: {frag} bp')

simulate_gel([vector_fragments], ['EcoRI + NotI'], title='Exercise 2: Virtual Digest')


### Exercise 3: ORF Finding in a Bacterial Genome Fragment

Find all ORFs ≥ 90 nt in the sequence below. Report:
1. Number of ORFs found in each of the 6 frames
2. The longest ORF and its translated protein
3. Average ORF length

**Tip**: A 90 nt ORF encodes a 30 aa protein — a minimal realistic threshold for bacteria.


In [ ]:
random.seed(77)
# Build a 2 kb fragment with two known ORFs embedded
bact_fragment = (
    ''.join(random.choices('ACGT', k=200))
    + 'ATG' + 'GCT' * 30 + 'CTG' * 10 + 'TAA'   # 126 nt ORF
    + ''.join(random.choices('ACGT', k=300))
    + 'ATG' + 'AAA' * 20 + 'GAT' * 15 + 'CAA' * 5 + 'TGA'   # 123 nt ORF
    + ''.join(random.choices('ACGT', k=200))
    + 'ATG' + 'CCG' * 50 + 'TAG'   # 153 nt ORF
    + ''.join(random.choices('ACGT', k=100))
)

# --- Solution ---
bact_orfs = find_orfs(bact_fragment, codon_table, min_length=90)

frame_counts = Counter((o['strand'], o['frame']) for o in bact_orfs)
total_len = sum(o['length_nt'] for o in bact_orfs)

print(f'Fragment length: {len(bact_fragment)} nt')
print(f'Total ORFs >= 90 nt: {len(bact_orfs)}')
print()
print('ORFs per frame:')
for (strand, frame), count in sorted(frame_counts.items()):
    print(f'  {strand}{frame}: {count} ORF(s)')
print()
if bact_orfs:
    longest = bact_orfs[0]
    print(f'Longest ORF: {longest["length_nt"]} nt on strand {longest["strand"]} frame {longest["frame"]}')
    print(f'Protein ({len(longest["protein"])} aa): {longest["protein"][:40]}...')
    print()
    avg_len = total_len / len(bact_orfs)
    print(f'Average ORF length: {avg_len:.1f} nt')


### Exercise 4: Three-Point Cross Problem

In *Drosophila*, a three-point cross between flies heterozygous for genes
**y** (yellow body), **m** (miniature wing), and **w** (white eye) yields the
following progeny:

| Genotype | Count |
|----------|-------|
| + + +    | 624   |
| y m w    | 618   |
| + m w    | 143   |
| y + +    | 151   |
| + + w    | 7     |
| y m +    | 9     |
| + m +    | 218   |
| y + w    | 230   |

Determine:
1. The gene order
2. Map distances between adjacent genes
3. Coefficient of coincidence and interference


In [ ]:
# Exercise 4 solution
drosophila_data = {
    '+++': 624, 'ymw': 618,  # parental
    '+mw': 143, 'y++': 151,  # single CO class 1
    '++w': 7,   'ym+': 9,    # double CO (smallest) -> middle gene identified
    '+m+': 218, 'y+w': 230,  # single CO class 2
}

total_fly = sum(drosophila_data.values())

# Double recombinants are '++w' and 'ym+'
# In parental: +++ and ymw -> so parental coupling is y-m-w (all together or all wild-type)
# Double recombinants swap the middle gene
# Parental: + + + and y m w
# DCO:      + + w and y m +  -> only w was exchanged, so w is in the MIDDLE
# Gene order: y --- w --- m  (or equivalently m --- w --- y)

dco_flies = drosophila_data['++w'] + drosophila_data['ym+']

# Region I: y to w  -> recombinants between y and w
# Single COs that separate y from w: '+mw' and 'y++'
# (these keep m on same side as original, and swap y/w relative to each other... )
# With order y-w-m:
#   Region I (y-w): exchange between y and w -> produces +++ <-> ymw  ...
#   Let's think systematically:
#   Parental haplotypes: y w m  /  + + +
#   SCO region I (y-w):  +  w m  /  y + +  -> matches 'y++' (143+151)
sco_region_I_yw = drosophila_data['+mw'] + drosophila_data['y++']
#   SCO region II (w-m): y + m / + w +  -> '+m+' and 'y+w' (218+230)
sco_region_II_wm = drosophila_data['+m+'] + drosophila_data['y+w']

dist_y_w = (sco_region_I_yw + dco_flies) / total_fly * 100
dist_w_m = (sco_region_II_wm + dco_flies) / total_fly * 100

expected_dco = (dist_y_w / 100) * (dist_w_m / 100) * total_fly
coc_fly = dco_flies / expected_dco
interference_fly = 1 - coc_fly

print('Drosophila three-point cross solution:')
print(f'  Total progeny: {total_fly}')
print(f'  Double recombinants: {dco_flies} ("++w" and "ym+")')
print(f'  => Middle gene is w (white eye)')
print(f'  Gene order: y --- w --- m')
print()
print(f'  Map distance y-w: {dist_y_w:.2f} cM')
print(f'  Map distance w-m: {dist_w_m:.2f} cM')
print()
print(f'  Expected DCOs: {expected_dco:.1f}')
print(f'  CoC: {coc_fly:.3f}')
print(f'  Interference: {interference_fly:.3f}')

draw_genetic_map(['y', 'w', 'm'], [dist_y_w, dist_w_m],
                 title='Drosophila Genetic Map (y — w — m)')


---

## Summary

| Concept | Key formula / tool | Typical range |
|---------|-------------------|---------------|
| Genetic code | 64 codons → 20 aa + 3 stops | 1–6 synonymous codons per aa |
| RSCU | $X_{ij} / \bar{X}_i$ | 0 (never used) to ~6 (always preferred) |
| CAI | geometric mean of $w$ values | 0 (random) to 1.0 (optimised) |
| GC3 | GC% at 3rd codon position | 12% (*Plasmodium*) to 70%+ (high-GC bacteria) |
| Restriction digest | cut at recognition palindrome | Type II: 4–8 bp sites |
| ORF finding | all 6 frames, min length filter | >300 nt for likely real genes |
| Recombination frequency | recombinants / total × 100 | 0–50 cM (max 50 for unlinked) |
| Coefficient of coincidence | observed DCO / expected DCO | 0 (complete interference) to >1 |
| Ts/Tv ratio | transitions / transversions | 2–4 (mammals), ~1 (expected by chance) |
| CpG O/E | observed CpG / expected CpG | <0.25 (depleted), >0.6 (CpG island) |
| Hardy-Weinberg | $p^2 + 2pq + q^2 = 1$ | χ² test df=1 |

---

[← Previous: Comparative Genomics](../../Tier_2_Core_Bioinformatics/12_Comparative_Genomics/01_comparative_genomics.ipynb) | [Next: Hi-C Analysis →](../14_Hi-C_Analysis/14_hic_analysis.ipynb)
